# Repository — exercises

Run `example.ipynb` first. Each exercise below re-declares the pieces it needs. Complete the code cells; do not look up a solution until you have tried the exercise.


## Exercise 1 — Extend the collection interface (basic application)

**Goal:** Add `find_by_email(email: str) -> User | None` to `UserRepository` and implement it on `InMemoryUserRepository`.

**Acceptance criteria:**
- After adding Ada (`ada@example.com`), `find_by_email('ada@example.com')` returns that user.
- Unknown emails return `None`.
- `display_name` is unchanged and still uses `get`.


In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass


@dataclass(frozen=True)
class User:
    id: str
    email: str
    name: str


class UserRepository(ABC):
    @abstractmethod
    def add(self, user: User) -> None:
        ...

    @abstractmethod
    def get(self, user_id: str) -> User | None:
        ...

    @abstractmethod
    def list_all(self) -> list[User]:
        ...
    
    @abstractmethod
    def find_by_email(self, email: str) -> User | None:
        ...

class InMemoryUserRepository(UserRepository):
    def __init__(self) -> None:
        self._by_id: dict[str, User] = {}

    def add(self, user: User) -> None:
        self._by_id[user.id] = user

    def get(self, user_id: str) -> User | None:
        return self._by_id.get(user_id)

    def list_all(self) -> list[User]:
        return list(self._by_id.values())
    
    def find_by_email(self, email: str) -> User | None:
        for user in self._by_id.values():
            if user.email == email:
                return user
        return None


def display_name(repo: UserRepository, user_id: str) -> str:
    user = repo.get(user_id)
    return 'unknown user' if user is None else user.name

def find_user_by_email(repo: UserRepository, email: str) -> User | None:
    return repo.find_by_email(email)


## Exercise 2 — A second concrete repository (adaptation)

**Goal:** Implement `ListUserRepository` that stores users in a `list` instead of a `dict`, still satisfying `UserRepository` (including `find_by_email` from Exercise 1 if you added it; otherwise implement `add` / `get` / `list_all`).

Show `display_name` working with both `InMemoryUserRepository` and `ListUserRepository` without changing `display_name`.

**Acceptance criteria:**
- Duplicate `id` values on `add` overwrite or replace the previous user (pick one rule and stick to it).
- `display_name(list_repo, 'u1')` prints `Ada Lovelace`.
- `display_name` is not edited.


In [2]:
from abc import ABC, abstractmethod
from dataclasses import dataclass


@dataclass(frozen=True)
class User:
    id: str
    email: str
    name: str


class UserRepository(ABC):
    @abstractmethod
    def add(self, user: User) -> None:
        ...

    @abstractmethod
    def get(self, user_id: str) -> User | None:
        ...

    @abstractmethod
    def list_all(self) -> list[User]:
        ...
    
    @abstractmethod
    def find_by_email(self, email: str) -> User | None:
        ...

class InMemoryUserRepository(UserRepository):
    def __init__(self) -> None:
        self._by_id: dict[str, User] = {}

    def add(self, user: User) -> None:
        self._by_id[user.id] = user

    def get(self, user_id: str) -> User | None:
        return self._by_id.get(user_id)

    def list_all(self) -> list[User]:
        return list(self._by_id.values())
    
    def find_by_email(self, email: str) -> User | None:
        for user in self._by_id.values():
            if user.email == email:
                return user
        return None

class ListUserRepository(UserRepository):
    def __init__(self) -> None:
        self._users: list[User] = []

    def add(self, user: User) -> None:
        # Replace an existing user with the same id.
        for index, existing_user in enumerate(self._users):
            if existing_user.id == user.id:
                self._users[index] = user
                return
        self._users.append(user)

    def get(self, user_id: str) -> User | None:
        for user in self._users:
            if user.id == user_id:
                return user
        return None

    def list_all(self) -> list[User]:
        return list(self._users)

    def find_by_email(self, email: str) -> User | None:
        for user in self._users:
            if user.email == email:
                return user
        return None

def display_name(repo: UserRepository, user_id: str) -> str:
    user = repo.get(user_id)
    return "unknown user" if user is None else user.name

ada = User(id="u1", email="ada@example.com", name="Ada Lovelace")

dict_repo = InMemoryUserRepository()
list_repo = ListUserRepository()

dict_repo.add(ada)
list_repo.add(ada)

print(display_name(dict_repo, "u1"))
print(display_name(list_repo, "u1"))

Ada Lovelace
Ada Lovelace


## Exercise 3 — Critique (design judgment)

In a markdown cell below, answer:

1. How is Repository different from Active Record if both can load a `User` by id? Use the example's `User` class in your answer.
2. A teammate wants `class Repository[T]` with `get`, `add`, and `query(sql: str)`. Does that still match the pattern? Why or why not?
3. When would you skip Repository and let a service call the database (or an ORM) directly?


1. With Active Record, the User object would contain persistence behavior, such as saving itself or loading a user by ID. In this example, User is a plain dataclass; UserRepository is a separate object that handles storing and retrieving users.

2. get and add can fit a Repository interface, but query(sql: str) exposes SQL to callers and couples them to the database. That weakens the repository’s role as an abstraction over persistence. Prefer methods that describe domain operations, such as find_by_email.

3. I might skip Repository in a small application with straightforward data access, especially if the ORM already provides a clear abstraction. If a separate repository adds boilerplate without making the code easier to understand, test, or change, a service can use the ORM or database directly.